# `langchain_protocol` smoke test

Python bindings for the LangChain agent streaming protocol. The package exports `TypedDict` classes,
`Literal` unions, and union aliases generated from the wire-format CDDL. There is no runtime client
or transport &mdash; only typing primitives &mdash; so the tests exercise structural usage of those types.

In [ ]:
# Example 1: build a SubscribeParams TypedDict and a wrapping Command, mirroring the README usage.
from langchain_protocol import Command, SubscribeParams

params: SubscribeParams = {
    "channels": ["messages", "lifecycle"],
}

subscribe: Command = {
    "id": 1,
    "method": "subscription.subscribe",
    "params": params,
}

print("command:", subscribe)
assert subscribe["method"] == "subscription.subscribe"
assert subscribe["params"]["channels"] == ["messages", "lifecycle"]
# TypedDicts are plain dicts at runtime
assert isinstance(subscribe, dict)
assert isinstance(params, dict)

In [ ]:
# Example 2: inspect a TypedDict's declared field schema via __annotations__.
from langchain_protocol import SubscribeParams, InputMessage, Checkpoint

for cls in (SubscribeParams, InputMessage, Checkpoint):
    fields = list(cls.__annotations__.keys())
    print(f"{cls.__name__}: {fields}")
    assert fields, f"{cls.__name__} should declare at least one field"

# The classes are TypedDicts -> subclasses of dict at runtime.
assert issubclass(SubscribeParams, dict)
assert issubclass(InputMessage, dict)
assert issubclass(Checkpoint, dict)

In [ ]:
# Example 3: the Literal-union aliases enumerate the protocol's enum values.
import typing

from langchain_protocol import Channel, ErrorCode

channel_values = set()
for arg in typing.get_args(Channel):
    lit_args = typing.get_args(arg)
    if lit_args and isinstance(lit_args[0], str):
        channel_values.add(lit_args[0])

error_codes = set()
for arg in typing.get_args(ErrorCode):
    lit_args = typing.get_args(arg)
    if lit_args and isinstance(lit_args[0], str):
        error_codes.add(lit_args[0])

print("channels:", sorted(channel_values))
print("error codes:", sorted(error_codes))
assert {"messages", "lifecycle", "updates"}.issubset(channel_values)
assert {"invalid_argument", "permission_denied"}.issubset(error_codes)

In [ ]:
# Example 4: Command is a tagged union of variant TypedDicts; enumerate them.
import typing

from langchain_protocol import Command

variants = typing.get_args(Command)
print(f"Command has {len(variants)} variant(s)")
for variant in variants:
    method_type = variant.__annotations__.get("method")
    method_literal = typing.get_args(method_type)
    method_name = method_literal[0] if method_literal else "<unknown>"
    print(f"  {variant.__name__}: method={method_name!r}")

assert len(variants) > 1, "Command should be a union of command variants"
for variant in variants:
    assert "method" in variant.__annotations__

In [ ]:
# Example 5: round-trip a couple of representative payloads through JSON serialization.
import json

from langchain_protocol import InputMessage, AgentTreeNode, Checkpoint

msg: InputMessage = {"role": "user", "content": "hello"}
node: AgentTreeNode = {
    "namespace": ["root"],
    "status": "idle",
    "graph_name": "main",
    "children": [],
}
ckpt: Checkpoint = {"id": "ckpt-1", "parent_id": None, "step": 0, "source": "input"}

for payload in (msg, node, ckpt):
    encoded = json.dumps(payload)
    decoded = json.loads(encoded)
    assert decoded == payload, f"round-trip mismatch for {payload}"
    print(f"ok: {encoded[:80]}")

print("\nlangchain_protocol smoke test passed")
assert True